In [14]:
import pandas as pd
import numpy as np


# df = pd.read_csv("./data/log_sensor.csv")
df = pd.read_csv("./data/sintetis/dataset_mentah.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 5000 baris, 10 kolom


,jam,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec,plant_age
0,0,55.0,27.6,25.2,78.9,62.7,86.9,104.9,2.50,0
1,1,52.2,26.1,24.6,80.2,64.5,91.1,102.7,2.33,0
2,2,50.5,25.8,24.3,76.8,62.9,92.3,107.9,2.25,0
3,3,49.6,25.0,24.6,79.5,62.1,92.0,110.8,2.24,0
4,4,49.4,26.5,23.9,78.9,63.5,88.0,114.1,2.23,0


# Cek data

In [15]:
df[["soil_moisture", "soil_temperature", "air_temperature", "air_humidity"]].describe().round(2)


,soil_moisture,soil_temperature,air_temperature,air_humidity
count,5000.00,5000.00,5000.00,5000.00
mean,69.41,27.00,27.99,71.98
std,10.71,1.47,2.95,5.90
min,29.20,23.90,21.90,58.30
25%,62.10,25.70,25.30,66.60
50%,69.70,27.00,28.00,72.10
75%,77.30,28.40,30.70,77.30
max,95.00,30.10,34.90,84.50


# Fungsi labelling

In [16]:
def label_irigasi(row):
    """
    Keputusan siram/tidak berbasis kelembaban tanah + konteks lingkungan.

    Referensi:
    - Threshold kelembaban 60-80% (Hatta 2006; skripsi Alvian Tabel 4.12)
    - Adaptif terhadap suhu & kelembaban udara (Rochmanto & Nursaputro, 2025)

    Logika:
    - Tanah kering (<60%) → siram
    - Tanah normal (60-80%) → default tidak, TAPI kalau panas + udara kering,
      siram lebih awal (antisipasi cepat kering)
    - Tanah basah (>80%) → jangan siram
    """
    sm = row["soil_moisture"]
    st = row["soil_temperature"]
    at = row["air_temperature"]
    ah = row["air_humidity"]

    # Threshold dasar: kering di bawah 60%
    threshold = 60

    # --- Adaptasi terhadap kondisi lingkungan ---
    # Suhu tanah tinggi → evaporasi cepat → naikkan threshold (siram lebih awal)
    if st > 30:
        threshold += 3

    # Suhu udara tinggi + kelembaban udara rendah → transpirasi tinggi
    if at > 32 and ah < 55:
        threshold += 4

    # Kelembaban udara sangat tinggi (baru hujan/lembab) → tunda, air bakal naik
    if ah > 85:
        threshold -= 5

    # Keputusan
    if sm >= 80:
        action = 0            # basah, jangan siram (cegah busuk akar)
    elif sm < threshold:
        action = 1            # kering (relatif kondisi) → siram
    else:
        action = 0            # normal → cukup

    return action

print("Fungsi label_irigasi() siap.")

Fungsi label_irigasi() siap.


# Terapkan labelling

In [17]:
df["irrigation_action"] = df.apply(label_irigasi, axis=1)

print("Distribusi label:")
dist = df["irrigation_action"].value_counts().sort_index()
for val, count in dist.items():
    label = "Tidak siram" if val == 0 else "Siram"
    print(f"  {val} ({label}): {count} ({count/len(df)*100:.1f}%)")


Distribusi label:
  0 (Tidak siram): 4067 (81.3%)
  1 (Siram): 933 (18.7%)


# Contoh

In [21]:
# Lihat baris yang keputusannya dipengaruhi faktor lingkungan (bukan cuma moisture)
sample = df[["soil_moisture", "soil_temperature", "air_temperature",
             "air_humidity", "irrigation_action"]].head(20)
sample

,soil_moisture,soil_temperature,air_temperature,air_humidity,irrigation_action
0,55.0,27.6,25.2,78.9,1
1,52.2,26.1,24.6,80.2,1
2,50.5,25.8,24.3,76.8,1
3,49.6,25.0,24.6,79.5,1
4,49.4,26.5,23.9,78.9,1
5,48.0,25.4,23.0,79.6,1
6,58.8,25.6,25.2,75.9,1
7,57.5,24.8,25.8,74.2,1
8,55.0,25.0,27.1,73.5,1
9,53.6,25.4,28.8,70.6,1


# Simpan

In [22]:
FEATURES = ["soil_moisture", "soil_temperature", "air_temperature", "air_humidity"]
cols = FEATURES + ["irrigation_action"]

df[cols].to_csv("data/dataset_irigasi.csv", index=False)
print("Tersimpan: data/dataset_irigasi.csv")
print(f"Total: {len(df)} baris, kolom: {cols}")


Tersimpan: data/dataset_irigasi.csv
Total: 5000 baris, kolom: ['soil_moisture', 'soil_temperature', 'air_temperature', 'air_humidity', 'irrigation_action']
